# Phase 1: Local chatbot with LangGraph and Ollama

This notebook builds a simple chatbot using the locally running Ollama model qwen3.5:2b and LangGraph. Run the cells from top to bottom.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict

from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

llm = ChatOllama(model="qwen3.5:2b", temperature=0)

class State(TypedDict):
    messages: Annotated[list, add_messages]

def chatbot(state: State):
    return {"messages": [llm.invoke(state["messages"])]}

graph_builder = StateGraph(State)
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)

graph = graph_builder.compile()

print("Graph created.")

In [ ]:
from IPython.display import display

display(
    {"text/vnd.mermaid": graph.get_graph().draw_mermaid()},
    raw=True,
)

In [ ]:
def stream_graph_updates(user_input: str):
    print("Assistant: ", end="", flush=True)

    for chunk, metadata in graph.stream(
        {"messages": [{"role": "user", "content": user_input}]},
        stream_mode="messages",
    ):
        if metadata.get("langgraph_node") == "chatbot" and chunk.content:
            print(chunk.content, end="", flush=True)

    print()

stream_graph_updates("What are large language models?")